# EcoHome Energy Advisor - Agent Run and Evaluation

In this notebook, we'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality, relevance, and tool use
- Check conversation memory on a follow-up question
- Score RAG answers with Ragas
- Identify areas for improvement


## 1. Import and Initialize


In [2]:
from datetime import datetime
from agent import Agent, DEFAULT_SYSTEM_INSTRUCTIONS
import os
import re
from typing import Dict, Any, List, Set

from dotenv import load_dotenv
load_dotenv()


/Users/andreymashukov/Desktop/projects/ecohome_solution/.venv/lib/python3.13/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


True

In [3]:
## Create the agent's instructions
# The default prompt in agent.py includes tool routing, date context,
# error handling, multi-device scheduling, answer-first relevance, and
# conversation-memory rules.

ECOHOME_SYSTEM_PROMPT = DEFAULT_SYSTEM_INSTRUCTIONS


In [4]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)
ecohome_agent.get_agent_tools()


['get_weather_forecast',
 'get_electricity_prices',
 'query_energy_usage',
 'query_solar_generation',
 'get_recent_energy_summary',
 'search_energy_tips',
 'calculate_energy_savings']

In [5]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)
print(response["messages"][-1].content)


To minimize costs and maximize solar power when charging your electric vehicle (EV) tomorrow (Wednesday, August 19), the best time to charge is during the early morning hours when solar irradiance starts to increase and electricity rates are low.

### Recommended Charging Time:
- **Charge your EV from 5 AM to 8 AM.**
  - **5 AM - 6 AM:** Electricity rate is **$0.132/kWh** (off-peak) and solar irradiance starts at **0 W/m²**.
  - **6 AM - 8 AM:** Electricity rate remains **$0.132/kWh** (off-peak) with solar irradiance starting to increase to **205 W/m²** by 8 AM.

### Summary of Rates:
- **5 AM - 6 AM:** $0.132/kWh
- **6 AM - 8 AM:** $0.132/kWh
- **8 AM - 9 AM:** $0.22/kWh (mid-peak)

### Actions:
1. **Charge your EV from 5 AM to 6 AM** to take advantage of the lowest rate of **$0.132/kWh**.
2. **Continue charging from 6 AM to 8 AM** as solar power begins to contribute, still at **$0.132/kWh**.
3. **Monitor solar generation** after 8 AM to maximize the use of solar energy for your EV ch

In [6]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump() if hasattr(msg, "model_dump") else msg
    if isinstance(obj, dict) and obj.get("tool_call_id"):
        print("-", getattr(msg, "name", obj.get("name")))
    tool_calls = obj.get("tool_calls") if isinstance(obj, dict) else None
    if tool_calls:
        for tc in tool_calls:
            print("- call", tc.get("name"))


TOOLS:
- call get_weather_forecast
- call get_electricity_prices
- get_weather_forecast
- get_electricity_prices


## 2. Define Test Cases


In [7]:
# Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "Must recommend specific charging hours, reference sunny periods and off-peak pricing.",
    },
    {
        "id": "thermostat_2",
        "question": "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "Should suggest a numeric thermostat range and explain price/weather reasoning.",
    },
    {
        "id": "dishwasher_3",
        "question": "How much can I save by running my dishwasher during off-peak hours?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "Must estimate savings per cycle and per month using TOU rate difference.",
    },
    {
        "id": "laundry_4",
        "question": "When is the best time to run my washing machine this weekend?",
        "expected_tools": ["get_electricity_prices"],
        "expected_response": "Should detect weekend pricing and recommend cheapest hours.",
    },
    {
        "id": "solar_forecast_5",
        "question": "How much solar energy can I expect tomorrow in San Francisco?",
        "expected_tools": ["get_weather_forecast"],
        "expected_response": "Should reference sunny hours, irradiance levels, or expected generation patterns.",
    },
    {
        "id": "usage_history_6",
        "question": "Suggest three ways I can reduce energy use based on my usage history.",
        "expected_tools": ["get_recent_energy_summary", "search_energy_tips"],
        "expected_response": "Must identify high-consumption devices and provide three concrete actions.",
    },
    {
        "id": "optimization_multi_device_7",
        "question": "Help me schedule my EV, dishwasher, and dryer tomorrow for lowest electricity cost.",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "Should propose a coordinated time schedule minimizing on-peak usage.",
    },
    {
        "id": "energy_tips_8",
        "question": "Give me three ways to reduce electricity usage at home.",
        "expected_tools": ["search_energy_tips"],
        "expected_response": "Should return 3 actionable, personalized efficiency recommendations.",
    },
    {
        "id": "recent_summary_9",
        "question": "Summarize my energy usage over the past 48 hours.",
        "expected_tools": ["get_recent_energy_summary"],
        "expected_response": "Should return total kWh, cost, device breakdown, and insights.",
    },
    {
        "id": "pool_pump_10",
        "question": "What's the best time to run my pool pump this week based on the weather forecast?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "Must recommend a daily schedule balancing sunlight and off-peak pricing.",
    },
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")


## 3. Run Agent Tests


In [8]:
CONTEXT = "Location: San Francisco, CA"


In [9]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

def _final_text(response) -> str:
    from agent import _final_answer
    return _final_answer(response)


print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)

    try:
        response = ecohome_agent.invoke(
            question=test_case["question"],
            context=CONTEXT,
            reset_history=True,
        )
        answer = _final_text(response)
        print(answer[:400])
        test_results.append(
            {
                "test_id": test_case["id"],
                "question": test_case["question"],
                "response": answer,
                "messages": response["messages"] if isinstance(response, dict) else [],
                "expected_tools": test_case["expected_tools"],
                "expected_response": test_case["expected_response"],
                "timestamp": datetime.now().isoformat(),
                "failed": False,
            }
        )
    except Exception as e:
        print(f"Error: {e}")
        test_results.append(
            {
                "test_id": test_case["id"],
                "question": test_case["question"],
                "response": f"Error: {str(e)}",
                "messages": [],
                "expected_tools": test_case["expected_tools"],
                "expected_response": test_case["expected_response"],
                "timestamp": datetime.now().isoformat(),
                "error": str(e),
                "failed": True,
            }
        )

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------
To minimize costs and maximize solar power when charging your electric car tomorrow (Wednesday, August 19), the best time to charge is during the hours when solar generation is high and electricity prices are low.

### Optimal Charging Time:
- **Best Time to Charge:** 8 AM to 10 AM
  - **Solar Irradiance:** 
    - 8 AM: 205 W/m²
    - 9 AM: (not provided, but expected to increase)
    - 10 AM: 514

Test 2: thermostat_2
Question: What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?
--------------------------------------------------
On Wednesday afternoon (August 19), electricity prices will spike to **$0.33 per kWh** from **4 PM to 7 PM**. Given the forecasted temperature of **15.1°C (59.2°F)** at **2 PM** and **14.4°C (57.9°F)** at **5 PM**, you 

## 4. Evaluate Responses


In [9]:
# Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage


def _normalize(text: str) -> str:
    if text is None:
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s:.%-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _tokenize(text: str) -> List[str]:
    return [t for t in _normalize(text).split(" ") if t]


def _overlap_score(a: str, b: str) -> float:
    ta = set(_tokenize(a))
    tb = set(_tokenize(b))
    if not ta or not tb:
        return 0.0
    inter = len(ta.intersection(tb))
    union = len(ta.union(tb))
    return inter / union if union > 0 else 0.0


def evaluate_response(question: str, final_response: str, expected_response: str) -> Dict[str, Any]:
    resp_norm = _normalize(final_response)
    q_norm = _normalize(question)
    if any(x in resp_norm for x in ["internal error", "try again"]) or not resp_norm:
        low = {"score": 0.0, "feedback": "The agent did not produce a usable answer."}
        return {
            "accuracy": low, "relevance": low, "completeness": low, "usefulness": low,
            "overall_score": 0.0, "overall_feedback": "Empty or error response.",
        }

    stop = {
        "what", "when", "where", "which", "who", "why", "how", "should", "would",
        "could", "can", "the", "and", "for", "with", "from", "this", "that",
        "based", "over", "past", "much", "many", "give", "three", "ways",
        "best", "time", "help", "suggest",
    }
    aliases = {
        "electric": ["ev", "tesla", "vehicle", "car"],
        "car": ["ev", "tesla", "vehicle"],
        "thermostat": ["setpoint", "hvac", "temperature"],
        "washing": ["washer", "laundry"],
        "machine": ["washer", "laundry"],
        "dishwasher": ["dishes"],
        "history": ["usage", "consumption", "kwh"],
        "summarize": ["summary", "total"],
        "temperature": ["setpoint", "thermostat", "f", "c"],
    }

    def _has_term(text: str, term: str) -> bool:
        if term in text:
            return True
        return any(alias in text for alias in aliases.get(term, []))

    expected_keys = [t for t in _tokenize(expected_response) if len(t) > 3]
    hits = sum(1 for t in expected_keys if t in resp_norm)
    coverage = hits / len(expected_keys) if expected_keys else 0.5

    question_keys = [t for t in _tokenize(question) if len(t) > 3 and t not in stop]
    q_hits = sum(1 for t in question_keys if _has_term(resp_norm, t))
    question_coverage = q_hits / len(question_keys) if question_keys else 0.5

    lead = str(final_response).strip().split("\n")[0]
    lead_overlap = _overlap_score(lead, question)

    has_numbers = bool(re.search(r"\d", resp_norm))
    has_money = "$" in str(final_response) or "usd" in resp_norm or "kwh" in resp_norm
    has_time = bool(re.search(r"\b(\d{1,2}\s?(am|pm)|off-peak|on-peak|mid-peak)\b", resp_norm))
    has_actions = any(kw in resp_norm for kw in ["recommend", "should", "schedule", "set", "run", "charge", "consider"])
    when_question = "when" in q_norm or "best time" in q_norm

    resp_len = len(resp_norm.split())
    brevity_bonus = 0.10 if 25 <= resp_len <= 220 else 0.0
    ramble_penalty = 0.15 if resp_len > 280 else 0.0
    direct_bonus = 0.12 if when_question and has_time else 0.0

    accuracy_raw = min(1.0, 0.45 + coverage * 0.35 + (0.1 if has_numbers else 0) + (0.1 if has_time or has_money else 0))
    relevance_raw = min(
        1.0,
        max(
            0.0,
            0.28
            + question_coverage * 0.42
            + lead_overlap * 0.15
            + (0.08 if has_actions else 0)
            + direct_bonus
            + brevity_bonus
            - ramble_penalty,
        ),
    )
    length_factor = 1.0 if resp_len >= 80 else 0.75 if resp_len >= 40 else 0.4
    completeness_raw = min(1.0, (0.5 + coverage * 0.4 + (0.1 if has_time or has_money else 0)) * length_factor)
    usefulness_raw = min(1.0, 0.4 + (0.2 if has_actions else 0) + (0.2 if has_numbers else 0) + (0.2 if has_time or has_money else 0))
    overall = (accuracy_raw + relevance_raw + completeness_raw + usefulness_raw) / 4.0

    def band(score, high, mid):
        if score >= 0.75:
            return high
        if score >= 0.4:
            return mid
        return "Needs improvement."

    return {
        "accuracy": {"score": round(accuracy_raw, 3), "feedback": band(accuracy_raw, "Aligns with expected guidance.", "Partial match to expected guidance.")},
        "relevance": {"score": round(relevance_raw, 3), "feedback": band(relevance_raw, "Stays on the question.", "Somewhat on topic.")},
        "completeness": {"score": round(completeness_raw, 3), "feedback": band(completeness_raw, "Covers the key expected details.", "Misses some expected details.")},
        "usefulness": {"score": round(usefulness_raw, 3), "feedback": band(usefulness_raw, "Actionable and specific.", "Could be more concrete.")},
        "overall_score": round(overall, 3),
        "overall_feedback": band(overall, "Strong response.", "Decent response with room to improve."),
    }



def _extract_tools_used(messages) -> List[str]:
    if messages is None:
        return []
    used: List[str] = []
    for msg in messages:
        obj = msg.model_dump() if hasattr(msg, "model_dump") else msg
        if isinstance(obj, dict):
            for tc in obj.get("tool_calls") or []:
                if tc.get("name"):
                    used.append(tc["name"])
            if obj.get("type") == "tool" and obj.get("name"):
                used.append(obj["name"])
        if getattr(msg, "tool_call_id", None) and getattr(msg, "name", None):
            used.append(msg.name)
    seen = set()
    unique = []
    for name in used:
        if name not in seen:
            seen.add(name)
            unique.append(name)
    return unique


def evaluate_tool_usage(messages, expected_tools: List[str]) -> Dict[str, Any]:
    expected_set = set(expected_tools or [])
    used_tools = _extract_tools_used(messages)
    used_set = set(used_tools)
    correct_used = used_set.intersection(expected_set)
    extra_used = used_set.difference(expected_set)
    missing = expected_set.difference(used_set)

    appropriateness_score = (len(correct_used) / len(used_set)) if used_tools else (1.0 if not expected_set else 0.0)
    completeness_score = (len(correct_used) / len(expected_set)) if expected_set else 1.0
    overall_tool_score = (appropriateness_score + completeness_score) / 2.0

    return {
        "used_tools": used_tools,
        "expected_tools": list(expected_set),
        "tool_appropriateness": {
            "score": round(appropriateness_score, 3),
            "feedback": f"Used {sorted(used_set) or ['<none>']}. Extra: {sorted(extra_used) or ['<none>']}.",
        },
        "tool_completeness": {
            "score": round(completeness_score, 3),
            "feedback": f"Missing: {sorted(missing) or ['<none>']}.",
        },
        "overall_score": round(overall_tool_score, 3),
        "overall_feedback": "Tool usage looks solid." if overall_tool_score >= 0.8 else "Tool usage could be more complete.",
    }


def generate_evaluation_report(test_results: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not test_results:
        return {"summary": {"num_tests": 0}, "per_test": [], "strengths": [], "weaknesses": [], "recommendations": []}

    totals = {
        "acc": 0.0, "rel": 0.0, "comp": 0.0, "use": 0.0,
        "resp": 0.0, "tool_a": 0.0, "tool_c": 0.0, "tool": 0.0,
    }
    num_failed = 0
    per_test = []

    for tr in test_results:
        failed = tr.get("failed", False)
        if failed:
            num_failed += 1
            resp_eval = {
                "accuracy": {"score": 0.0, "feedback": "Failed."},
                "relevance": {"score": 0.0, "feedback": "Failed."},
                "completeness": {"score": 0.0, "feedback": "Failed."},
                "usefulness": {"score": 0.0, "feedback": "Failed."},
                "overall_score": 0.0,
                "overall_feedback": "Agent execution failed.",
            }
            tool_eval = {
                "used_tools": [],
                "expected_tools": tr.get("expected_tools", []),
                "tool_appropriateness": {"score": 0.0, "feedback": "No tools used."},
                "tool_completeness": {"score": 0.0, "feedback": "Expected tools missing."},
                "overall_score": 0.0,
                "overall_feedback": "No tool execution.",
            }
        else:
            resp_eval = evaluate_response(tr.get("question", ""), str(tr.get("response", "")), tr.get("expected_response", ""))
            tool_eval = evaluate_tool_usage(tr.get("messages", []), tr.get("expected_tools", []))

        totals["acc"] += resp_eval["accuracy"]["score"]
        totals["rel"] += resp_eval["relevance"]["score"]
        totals["comp"] += resp_eval["completeness"]["score"]
        totals["use"] += resp_eval["usefulness"]["score"]
        totals["resp"] += resp_eval["overall_score"]
        totals["tool_a"] += tool_eval["tool_appropriateness"]["score"]
        totals["tool_c"] += tool_eval["tool_completeness"]["score"]
        totals["tool"] += tool_eval["overall_score"]
        per_test.append(
            {
                "test_id": tr.get("test_id"),
                "question": tr.get("question"),
                "response_snippet": str(tr.get("response", ""))[:280],
                "response_metrics": resp_eval,
                "tool_metrics": tool_eval,
            }
        )

    n = len(test_results)
    avg = {k: round(v / n, 2) for k, v in totals.items()}
    strengths, weaknesses, recommendations = [], [], []
    if avg["rel"] > 0.7:
        strengths.append("Responses stay aligned with the user questions.")
    else:
        weaknesses.append("Relevance is mixed. Some answers drift.")
        recommendations.append("Keep the first paragraph as a direct answer to the asked question.")
    if avg["comp"] > 0.7:
        strengths.append("Most answers include the expected details.")
    else:
        weaknesses.append("Some answers miss hours, cost, or solar details.")
        recommendations.append("Require time windows, prices, and solar notes in scheduling answers.")
    if avg["tool_a"] > 0.7:
        strengths.append("The agent usually selects the right tools.")
    else:
        weaknesses.append("Tool selection is inconsistent.")
        recommendations.append("Keep explicit tool routing in the system prompt.")
    if avg["tool_c"] < 0.7:
        weaknesses.append("Not all expected tools are used in multi-tool cases.")
        recommendations.append("For scheduling, always combine weather and pricing tools.")
    if num_failed:
        weaknesses.append(f"{num_failed} test(s) failed.")
        recommendations.append("Keep tool and agent error handling in place.")

    return {
        "summary": {
            "num_tests": n,
            "num_failed": num_failed,
            "avg_accuracy": avg["acc"],
            "avg_relevance": avg["rel"],
            "avg_completeness": avg["comp"],
            "avg_usefulness": avg["use"],
            "avg_overall_response_score": avg["resp"],
            "avg_tool_appropriateness": avg["tool_a"],
            "avg_tool_completeness": avg["tool_c"],
            "avg_overall_tool_score": avg["tool"],
        },
        "per_test": per_test,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recommendations,
    }


def display_evaluation_report(report: Dict[str, Any]) -> None:
    summary = report.get("summary", {})
    print("====================================================")
    print("      EcoHome Energy Advisor - Evaluation Report    ")
    print("====================================================\n")
    print("=== Summary Metrics ===")
    print(f"Total tests        : {summary.get('num_tests', 0)}")
    print(f"Failed tests       : {summary.get('num_failed', 0)}")
    print(f"Avg Accuracy       : {summary.get('avg_accuracy', 0.0):.2f}")
    print(f"Avg Relevance      : {summary.get('avg_relevance', 0.0):.2f}")
    print(f"Avg Completeness   : {summary.get('avg_completeness', 0.0):.2f}")
    print(f"Avg Usefulness     : {summary.get('avg_usefulness', 0.0):.2f}")
    print(f"Avg Resp. Overall  : {summary.get('avg_overall_response_score', 0.0):.2f}")
    print(f"Avg Tool Appropri. : {summary.get('avg_tool_appropriateness', 0.0):.2f}")
    print(f"Avg Tool Completeness: {summary.get('avg_tool_completeness', 0.0):.2f}")
    print(f"Avg Tool Overall   : {summary.get('avg_overall_tool_score', 0.0):.2f}\n")

    print("=== Per-Test Details  ===")
    for t in report.get("per_test", []):
        rm = t["response_metrics"]
        tm = t["tool_metrics"]
        print(f"- Test ID  : {t['test_id']}")
        print(f"  Question : {t['question']}")
        print(f"  Response : {t['response_snippet']!r}")
        print(
            f"  Response Scores -> acc={rm['accuracy']['score']:.2f}, rel={rm['relevance']['score']:.2f}, "
            f"comp={rm['completeness']['score']:.2f}, use={rm['usefulness']['score']:.2f}, overall={rm['overall_score']:.2f}"
        )
        print(
            f"  Tool Scores     -> appr={tm['tool_appropriateness']['score']:.2f}, "
            f"comp={tm['tool_completeness']['score']:.2f}, overall={tm['overall_score']:.2f}"
        )
        print(f"  Tools used     : {tm.get('used_tools')}\n")

    print("=== Strengths ===")
    for item in report.get("strengths") or ["(None identified yet)"]:
        print(f"- {item}")
    print("\n=== Weaknesses ===")
    for item in report.get("weaknesses") or ["(None identified yet)"]:
        print(f"- {item}")
    print("\n=== Recommendations ===")
    for item in report.get("recommendations") or ["(No specific recommendations yet)"]:
        print(f"- {item}")


## 4b. Multi-turn memory

The 10 scenario tests reset history so they stay independent. This cell checks that a follow-up can reuse a device and a constraint from the previous turn.


In [ ]:
def evaluate_memory_followup(turn1_question: str, turn2_question: str, turn2_answer: str) -> Dict[str, Any]:
    """Check that the follow-up reused the Tesla and the 4-9 PM constraint."""
    answer = _normalize(turn2_answer)
    remembers_device = any(token in answer for token in ["tesla", "model 3", "model3", "ev", "electric"])
    remembers_window = any(
        phrase in answer
        for phrase in ["4-9", "4 9", "4 pm", "4pm", "16:00", "peak"]
    )
    avoids_peak_as_best = not bool(
        re.search(r"best time[^\n]{0,80}(4\s*(pm|p\.m)|5\s*(pm|p\.m)|6\s*(pm|p\.m))", answer)
    )
    passed = remembers_device and remembers_window and avoids_peak_as_best
    return {
        "passed": passed,
        "remembers_device": remembers_device,
        "remembers_peak_constraint": remembers_window,
        "avoids_peak_as_best_window": avoids_peak_as_best,
        "turn1_question": turn1_question,
        "turn2_question": turn2_question,
        "turn2_snippet": str(turn2_answer)[:400],
    }


print("=== Multi-turn memory test ===")
turn1 = (
    "I have a Tesla Model 3. I do not want to charge during 4-9 PM peak hours. "
    "Please remember that preference."
)
turn2 = "When should I charge it tomorrow to minimize cost?"

memory_agent = Agent(instructions=ECOHOME_SYSTEM_PROMPT)
turn1_response = memory_agent.invoke(question=turn1, context=CONTEXT, reset_history=True)
turn1_text = _final_text(turn1_response)
print("Turn 1:", turn1)
print(str(turn1_text)[:350])
print()

turn2_response = memory_agent.invoke(question=turn2, context=CONTEXT, reset_history=False)
turn2_text = _final_text(turn2_response)
print("Turn 2:", turn2)
print(str(turn2_text)[:500])
print()

memory_result = evaluate_memory_followup(turn1, turn2, turn2_text)
print("Memory checks:")
for key in ("passed", "remembers_device", "remembers_peak_constraint", "avoids_peak_as_best_window"):
    print(f"  {key}: {memory_result[key]}")
if not memory_result["passed"]:
    print("Follow-up did not clearly reuse the Tesla or the 4-9 PM constraint.")


In [10]:
report = generate_evaluation_report(test_results)
display_evaluation_report(report)

print("\n=== Multi-turn memory ===")
if "memory_result" in globals():
    print(f"Passed: {memory_result['passed']}")
    print(f"Remembers device: {memory_result['remembers_device']}")
    print(f"Remembers 4-9 PM constraint: {memory_result['remembers_peak_constraint']}")
    print(f"Avoids peak as best window: {memory_result['avoids_peak_as_best_window']}")
else:
    print("Run the multi-turn memory cell first.")


      EcoHome Energy Advisor - Evaluation Report    

=== Summary Metrics ===
Total tests        : 10
Failed tests       : 0
Avg Accuracy       : 0.78
Avg Relevance      : 0.63
Avg Completeness   : 0.75
Avg Usefulness     : 1.00
Avg Resp. Overall  : 0.79
Avg Tool Appropri. : 0.95
Avg Tool Completeness: 1.00
Avg Tool Overall   : 0.97

=== Per-Test Details  ===
- Test ID  : ev_charging_1
  Question : When should I charge my electric car tomorrow to minimize cost and maximize solar power?
  Response : '### Best Time to Charge Your EV Tomorrow (2026-08-18)\n\n#### Solar Power Availability:\n- The solar irradiance is highest between **7 AM and 4 PM**, peaking around **12 PM** with 285 W/m². However, the sky is forecasted to be cloudy, which may reduce solar generation efficiency.\n\n#'
  Response Scores -> acc=0.79, rel=0.63, comp=0.76, use=1.00, overall=0.80
  Tool Scores     -> appr=1.00, comp=1.00, overall=1.00
  Tools used     : ['get_weather_forecast', 'get_electricity_prices']

- Test

## 5. Ragas RAG evaluation

Ragas scores a small grounded set: retrieval context precision plus answer relevancy / faithfulness for the two RAG-style agent questions.


In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.dataset_schema import EvaluationDataset, SingleTurnSample
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import AnswerRelevancy, ContextPrecision, Faithfulness

from tools import search_energy_tips


def _ragas_wrappers():
    api_key = os.getenv("OPENAI_API_KEY")
    llm_kwargs = {
        "model": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        "temperature": 0.0,
        "api_key": api_key,
        "max_retries": 2,
    }
    emb_kwargs = {"model": os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")}
    base_url = os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE")
    if base_url:
        llm_kwargs["base_url"] = base_url
        emb_kwargs["base_url"] = base_url
    if api_key:
        emb_kwargs["api_key"] = api_key
    return (
        LangchainLLMWrapper(ChatOpenAI(**llm_kwargs)),
        LangchainEmbeddingsWrapper(OpenAIEmbeddings(**emb_kwargs)),
    )


rag_gold = [
    {
        "question": "When should I charge an electric vehicle to save money?",
        "reference": (
            "Charge the EV during off-peak hours or when solar generation is high. "
            "Avoid 4-9 PM peak rates. Delayed start so charging finishes just before departure is usually cheaper."
        ),
        "search_query": "electric vehicle charging off-peak",
    },
    {
        "question": "How should I set a thermostat when electricity prices spike?",
        "reference": (
            "Pre-cool or pre-heat before peak hours, then relax the setpoint during the spike. "
            "Heating and cooling often account for 40-50% of home electricity use."
        ),
        "search_query": "thermostat HVAC peak prices",
    },
    {
        "question": "How can a home battery lower my bill?",
        "reference": (
            "Store cheap or solar energy and discharge during expensive peak hours. "
            "Do not charge the EV and the house battery from the grid at the same time during peak."
        ),
        "search_query": "home battery storage optimization",
    },
    {
        "question": "When should I run the dishwasher?",
        "reference": (
            "Run the dishwasher only when full, use eco mode, and schedule it for off-peak "
            "or high-solar hours with delayed start."
        ),
        "search_query": "dishwasher off-peak delayed start",
    },
]

samples = []
for item in rag_gold:
    retrieved = search_energy_tips.invoke({"query": item["search_query"], "max_results": 3})
    contexts = [tip["content"] for tip in retrieved.get("tips", [])] if "error" not in retrieved else []
    samples.append(
        SingleTurnSample(
            user_input=item["question"],
            retrieved_contexts=contexts,
            response=item["reference"],
            reference=item["reference"],
        )
    )

for test_id in ("energy_tips_8", "usage_history_6"):
    row = next((t for t in test_results if t.get("test_id") == test_id), None)
    if not row or row.get("failed"):
        continue
    retrieved = search_energy_tips.invoke({"query": row["question"], "max_results": 3})
    contexts = [tip["content"] for tip in retrieved.get("tips", [])] if "error" not in retrieved else []
    samples.append(
        SingleTurnSample(
            user_input=row["question"],
            retrieved_contexts=contexts,
            response=str(row["response"]),
            reference=row.get("expected_response") or "",
        )
    )

try:
    ragas_llm, ragas_embeddings = _ragas_wrappers()
    ragas_dataset = EvaluationDataset(samples=samples)
    ragas_result = evaluate(
        dataset=ragas_dataset,
        metrics=[
            ContextPrecision(),
            Faithfulness(),
            AnswerRelevancy(),
        ],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=False,
        show_progress=True,
    )
    ragas_df = ragas_result.to_pandas()
    score_cols = [c for c in ragas_df.columns if c in {"context_precision", "faithfulness", "answer_relevancy"}]
    print("=== Ragas scores ===")
    print(ragas_df[["user_input"] + score_cols].to_string(index=False))
    print("\nAverages:")
    for col in score_cols:
        print(f"  {col}: {ragas_df[col].mean():.2f}")
except Exception as e:
    ragas_result = None
    print(f"Ragas evaluation skipped: {type(e).__name__}: {e}")


## How to read the report

The scenario scores are heuristic. They reward:
- a first sentence that answers the question
- numeric hours, temperatures, and dollar estimates
- answers that stay on the named devices (not a long weather dump)
- the expected tools for each scenario

Ragas then scores retrieval and grounded answers with context precision, faithfulness, and answer relevancy.

The multi-turn cell checks that a follow-up still knows the Tesla and the 4-9 PM constraint.

Tool completeness is the most important automated check. If a scheduling question did not call both weather and pricing, tighten the system prompt and rerun.


## Evaluation summary from this run

10 independent scenarios plus a multi-turn memory check and a Ragas RAG score.

Previous local run (before relevance/memory/Ragas follow-ups):

- Avg tool completeness: 1.00
- Avg tool appropriateness: 0.95
- Avg tool overall: 0.97
- Avg response overall: 0.79
- Avg usefulness: 1.00

Re-run sections 3-5 after the prompt change to refresh heuristic relevance, memory, and Ragas numbers.
